# Rayleigh Quotient Bounds

Chapter 7: Symmetric Matrix · *Mathematics for AI and Machine Learning* by Fuheng Wu

See how x^T A x / x^T x stays between minimum and maximum eigenvalues.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-7/rayleigh-bounds/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's default symmetric matrix and probe angle (35 degrees)
A = torch.tensor([[2.1, 0.5],
                  [0.5, 0.8]], dtype=torch.float64)
theta = math.radians(35)


def rayleigh(A, x):
    return (x @ A @ x / (x @ x)).item()


x = torch.tensor([math.cos(theta), math.sin(theta)], dtype=torch.float64)
lmin, lmax = torch.linalg.eigvalsh(A).tolist()
r = rayleigh(A, x)
print(f"R(x) at 35 deg = {r:.6f}")
print(f"lambda_min = {lmin:.6f} <= R(x) <= lambda_max = {lmax:.6f}:", lmin <= r <= lmax)

# ── Sweep every direction: the extremes are exactly the eigenvalues ─────────
angles = torch.linspace(0, math.pi, 3601, dtype=torch.float64)
U = torch.stack([angles.cos(), angles.sin()], dim=1)
R = (U @ A * U).sum(dim=1)                       # unit vectors, so no division
print("\nmax R over directions =", R.max().item(), " at", math.degrees(angles[R.argmax()]), "deg")
print("min R over directions =", R.min().item(), " at", math.degrees(angles[R.argmin()]), "deg")
_, Q = torch.linalg.eigh(A)
print("eigenvector angles (deg):", [round(math.degrees(math.atan2(q[1], q[0])) % 180, 2) for q in Q.T])

# ── Maximizing R by gradient ascent on the unit circle finds lambda_max ─────
x = torch.tensor([1.0, -1.0], dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([x], lr=0.1)
for _ in range(300):
    opt.zero_grad()
    (-(x @ A @ x) / (x @ x)).backward()
    opt.step()
print("\ngradient ascent: R =", rayleigh(A, x.detach()), "(lambda_max =", lmax, ")")

R(x) at 35 deg = 2.142159
lambda_min = 0.629939 <= R(x) <= lambda_max = 2.270061: True

max R over directions = 2.270060850131508  at 18.8 deg
min R over directions = 0.6299391498684923  at 108.8 deg
eigenvector angles (deg): [108.78, 18.78]



gradient ascent: R = 2.270060973342836 (lambda_max = 2.2700609733428365 )
